# Lesson 07 - ਯੋਜਨਾ ਡਿਜ਼ਾਈਨ ਪੈਟਰਨ

ਇਹ ਨੋਟਬੁੱਕ Microsoft Agent Framework ਦੀ ਵਰਤੋਂ ਕਰਦਿਆਂ AI ਏਜੰਟਾਂ ਲਈ **ਯੋਜਨਾ ਡਿਜ਼ਾਈਨ ਪੈਟਰਨ** ਨੂੰ ਦਰਸਾਉਂਦਾ ਹੈ।
ਤੁਸੀਂ ਸਿੱਖੋਗੇ ਕਿ ਕਿਵੇਂ ਜਟਿਲ ਯਾਤਰਾ ਬੇਨਤੀਆਂ ਨੂੰ ਬਣਤਰਬੱਧ ਛੋਟੇ ਕਾਰਜਾਂ ਵਿੱਚ ਵੰਡਣਾ ਹੈ, ਉਨ੍ਹਾਂ ਨੂੰ ਵਿਸ਼ੇਸ਼ਗਿਆ ਏਜੰਟਾਂ ਨੂੰ ਸਮੰਪਿਤ ਕਰਨਾ ਹੈ,
ਅਤੇ ਨਤੀਜੇ ਵਜੋਂ ਬਨਾਈ ਗਈ ਯੋਜਨਾ ਨੂੰ ਲਾਗੂ ਕਰਨਾ ਹੈ — ਇਹ ਸਭ ਕੁਝ Pydantic ਮਾਡਲਾਂ ਦੁਆਰਾ ਸਮਰਥਿਤ ਬਣਤਰਬੱਧ ਨਤੀਜਾ ਵਰਤ ਕੇ।


## ਸੈੱਟਅੱਪ


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os, asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## ਟਾਸਕ ਵਿਭਾਜਨ

ਟਾਸਕ ਵਿਭਾਜਨ ਯੋਜਨਾ ਡਿਜ਼ਾਈਨ ਪੈਟਰਨ ਦਾ ਮੁੱਖ ਹਿੱਸਾ ਹੁੰਦਾ ਹੈ। ਇੱਕ ਹੀ ਏਜੰਟ ਨੂੰ ਸਭ ਕੁਝ ਸੰਭਾਲਣ ਦੇ ਬਜਾਏ, ਅਸੀਂ ਸਮੱਸਿਆ ਨੂੰ ਛੋਟੇ, ਵਧੀਆ ਪਰਿਭਾਸ਼ਿਤ **ਉਪ-ਟਾਸਕਾਂ** ਵਿੱਚ ਵੰਡਦੇ ਹਾਂ। ਹਰ ਉਪ-ਟਾਸਕ ਨੂੰ ਇੱਕ ਵਿਸ਼ੇਸ਼ਜ੍ਞ ਏਜੰਟ (ਜਿਵੇਂ ਕਿ ਉਡਾਣਾਂ, ਹੋਟਲ, ਸਰਗਰਮੀਆਂ) ਨਿਯੁਕਤ ਕੀਤਾ ਜਾਂਦਾ ਹੈ ਜਿਸਦੇ ਸਪਸ਼ਟ ਪ੍ਰਧਾਨਤਾ ਅਤੇ ਨਿਰਭਰਤਾ ਕ੍ਰਮ ਹੁੰਦੇ ਹਨ।

ਇਹ ਢੰਗ ਕਈ ਫਾਇਦੇ ਪ੍ਰਦਾਨ ਕਰਦਾ ਹੈ:
- **ਸਪਸ਼ਟਤਾ**: ਹਰ ਉਪ-ਟਾਸਕ ਦੀ ਇੱਕ ਸਿੰਗਲ ਜ਼ਿੰਮੇਵਾਰੀ ਹੁੰਦੀ ਹੈ।
- **ਸਮਾਂਤਰਤਾ**: ਸਵਤੰਤਰ ਉਪ-ਟਾਸਕ ਇਕੱਠੇ ਚੱਲ ਸਕਦੇ ਹਨ।
- **ਭਰੋਸਾ ਯੋਗਤਾ**: ਨਾਕਾਮੀਆਂ ਸਿਰਫ਼ ਵਿਅਕਤੀਗਤ ਉਪ-ਟਾਸਕਾਂ ਤੱਕ ਸੀਮਿਤ ਰਹਿੰਦੀਆਂ ਹਨ।
- **ਬਜਟ ਟ੍ਰੈਕਿੰਗ**: ਲਾਗਤਾਂ ਹਰ ਉਪ-ਟਾਸਕ ਲਈ ਅਨੁਮਾਨਿਤ ਕੀਤੀਆਂ ਜਾਂਦੀਆਂ ਹਨ ਅਤੇ ਜੋੜੀਆਂ ਜਾਂਦੀਆਂ ਹਨ।


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## ਬਣਤਰ ਵਾਲੇ ਨਤੀਜੇ ਨਾਲ ਯੋਜਨਾ ਬਨਾਉਣ ਵਾਲਾ ਏਜੰਟ ਬਣਾਉਣਾ

ਯੋਜਨਾ ਬਨਾਉਣ ਵਾਲਾ ਏਜੰਟ ਇੱਕ **ਫਰੰਟ ਡੇਸਕ ਕੋਆਰਡੀਨੇਟਰ** ਵਜੋਂ ਕੰਮ ਕਰਦਾ ਹੈ। ਇੱਕ ਉੱਚ-ਸਤਰ ਦੀ ਯਾਤਰਾ ਦੀ ਬੇਨਤੀ ਮਿਲਣ 'ਤੇ ਇਹ ਇੱਕ ਬਣਤਰਵਾਦ `TravelPlan` ਤਿਆਰ ਕਰਦਾ ਹੈ — ਬੇਨਤੀ ਨੂੰ ਉਪ-ਕਾਰਜਾਂ ਵਿੱਚ ਵੰਡਦਿਆਂ, ਪ੍ਰਾਥਮਿਕਤਾਵਾਂ ਸੈੱਟ ਕਰਦਿਆਂ, ਅਤੇ ਨਿਰਭਰਤਾਵਾਂ ਦੀ ਪਛਾਣ ਕਰਦਿਆਂ ਤਾਂ ਜੋ ਇੱਕ ਕੌਂਸੀਅਰਜ ਜਾਂ ਕਾਰਜਨਵਾਇਸੀ ਤਹਿੱਲਾ ਕੰਮ ਕਰ ਸਕੇ।


In [ ]:
planning_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    )
if result:
    plan = result
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## ਵਿਸ਼ੇਸ਼ਗੀ ਸੰਦਾਂ ਨਾਲ ਇੱਕ ਯੋਜਨਾ ਨੂੰ ਲਾਗੂ ਕਰਨਾ

ਜਦੋਂ ਫਰੰਟ ਡੈਸਕ ਏਜੰਟ ਨੇ ਇੱਕ ਸੰਰਚਿਤ ਯੋਜਨਾ ਤਿਆਰ ਕਰ ਲਈ ਹੁੰਦੀ ਹੈ, ਤਾਂ **ਕੰਸੀਅਰਜ ਏਜੰਟ** ਉਸਨੂੰ ਲਾਗੂ ਕਰਦਾ ਹੈ। 
ਹਰ ਵਿਸ਼ੇਸ਼ਗੀ ਸੰਦ ਇੱਕ ਉਪਕਾਰਜ ਦੇ ਇੱਕ ਸ਼੍ਰੇਣੀ ਦੀ ਸੰਭਾਲ ਕਰਦਾ ਹੈ (ਉਡਾਣਾਂ, ਹੋਟਲ, ਗਤੀਵਿਧੀਆਂ)। ਕੰਸੀਅਰਜ ਯੋਜਨਾ ਵਿੱਚ ਉਪਕਾਰਜਾਂ ਨੂੰ ਨਿਰਭਰਤਾ ਕ੍ਰਮ ਵਿੱਚ ਦੁਹਰਾਉਂਦਾ ਹੈ ਅਤੇ ਹਰ ਇੱਕ ਨੂੰ ਮੋਹਤਾਜ਼ ਸੰਦ ਵੱਲ ਭੇਜਦਾ ਹੈ।


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = await provider.create_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## Summary

ਇਸ ਪਾਠ ਵਿੱਚ ਤੁਸੀਂ AI ਏਜੰਟਾਂ ਲਈ **ਪਲਾਨਿੰਗ ਡਿਜ਼ਾਈਨ ਪੈਟਰਨ** ਸਿੱਖਿਆ:

1. **ਟਾਸਕ ਡੀਕੰਪੋਜ਼ੀਸ਼ਨ** — ਇੱਕ ਫਰੰਟ ਡੈਸਕ ਪਲਾਨਿੰਗ ਏਜੰਟ ਇੱਕ ਜਟਿਲ ਯਾਤਰਾ ਦੀ ਬੇਨਤੀ ਨੂੰ Pydantic ਮਾਡਲਾਂ ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਸੰਰਚਿਤ ਉਪਕਾਰਜਾਂ ਵਿੱਚ ਵੰਡਦਾ ਹੈ, ਹਰ ਇੱਕ ਨੂੰ ਪ੍ਰਾਥਮਿਕਤਾਵਾਂ ਅਤੇ ਨਿਰਭਰਤਾਵਾਂ ਦੇ ਨਾਲ ਇੱਕ ਵਿਸ਼ੇਸ਼ਗਿਆ ਏਜੰਟ ਨੂੰ ਸੌਂਪਦਾ ਹੈ।
2. **ਸੰਰਚਿਤ ਆਉਟਪੁੱਟ** — `response_format` ਦੇ ਕੇ ਏਜੰਟ ਇੱਕ ਮਾਨਤ TravelPlan ਵਸਤੂ ਮੁੜ ਪ੍ਰਦਾਨ ਕਰਦਾ ਹੈ ਨਾ ਕਿ ਮੁਫ਼ਤ-ਫਾਰਮ ਟੈਕਸਟ, ਜਿਸ ਨਾਲ ਅਗਲੇ ਪ੍ਰਕਿਰਿਆ ਭਰੋਸੇਯੋਗ ਹੋ ਜਾਂਦੀ ਹੈ।
3. **ਪਲਾਨ ਕਾਰਜਨਵਈ** — ਇੱਕ ਕਾਂਸੀਅਰਜ ਏਜੰਟ ਵਿਸ਼ੇਸ਼ਗਿਆ ਟੂਲਾਂ (`book_flight`, `reserve_hotel`, `book_activity`) ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਉਪਕਾਰਜਾਂ ਵਿੱਚ ਦੌੜਦਾ ਹੈ ਤਾ ਕਿ ਯੋਜਨਾ ਨੂੰ ਕਾਰਜਨਵਈ ਕਰਕੇ ਨਤੀਜੇ ਰਿਪੋਰਟ ਕਰ ਸਕੇ।

ਇਹ ਪੈਟਰਨ *ਕਿ ਕਰਨਾ ਹੈ* (ਪਲਾਨਿੰਗ) ਨੂੰ *ਕਿਵੇਂ ਕਰਨਾ ਹੈ* (ਕਾਰੀਵਾਈ) ਤੋਂ ਵੱਖ ਕਰਦਾ ਹੈ, ਜਿਸ ਨਾਲ ਏਜੰਟ ਜ਼ਿਆਦਾ ਮਾਡਯੂਲਰ, ਟੈਸਟੇਬਲ ਅਤੇ ਅਸਾਨੀ ਨਾਲ ਵਧਾਉਣ ਯੋਗ ਬਣ ਜਾਂਦੇ ਹਨ।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ਅਸਵੀਕਾਰੋੱਧੀ**:  
ਇਹ ਦਸਤਾਵੇਜ਼ AI ਅਨੁਵਾਦ ਸੇਵਾ [Co-op Translator](https://github.com/Azure/co-op-translator) ਦੁਆਰਾ ਅਨੁਵਾਦਿਤ ਕੀਤਾ ਗਿਆ ਹੈ। ਜਦੋਂ ਕਿ ਅਸੀਂ ਸਹੀਤਾ ਲਈ ਕੋਸ਼ਿਸ਼ ਕਰਦੇ ਹਾਂ, ਕਿਰਪਾ ਕਰਕੇ ਧਿਆਨ ਵਿੱਚ ਰੱਖੋ ਕਿ ਆਟੋਮੈਟਿਕ ਅਨੁਵਾਦਾਂ ਵਿੱਚ ਗਲਤੀਆਂ ਜਾਂ ਅਣਸਹੀਤੀਆਂ ਹੋ ਸਕਦੀਆਂ ਹਨ। ਮੂਲ ਦਸਤਾਵੇਜ਼ ਆਪਣੀ ਮੂਲ ਭਾਸ਼ਾ ਵਿੱਚ ਅਧਿਕਾਰਤ ਸਰੋਤ ਮੰਨਿਆ ਜਾਣਾ ਚਾਹੀਦਾ ਹੈ। ਮਹੱਤਵਪੂਰਨ ਜਾਣਕਾਰੀ ਲਈ, ਪੇਸ਼ਾਵਰ ਇਨਸਾਨੀ ਅਨੁਵਾਦ ਦੀ ਸਿਫਾਰਿਸ਼ ਕੀਤੀ ਜਾਂਦੀ ਹੈ। ਸਾਡੇ ਕੋਲ ਇਸ ਅਨੁਵਾਦ ਦੇ ਵਪਾਰਕ ਜਾਂ ਵਿਆਖਿਆਤਮਕ ਗਲਤ ਫਹਿਮੀਆਂ ਲਈ ਕੋਈ ਜ਼ਿੰਮੇਵਾਰੀ ਨਹੀਂ ਹੈ।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
